# WanGP — HER BİLGİSAYARDAN KULLANIM (Türkçe)

**Yerel Windows kurulumu yok.** Her bilgisayarda aynı Google hesabıyla bu defteri aç. Google Colab'daki GPU video üretir, **Google Drive kalıcı dosyaları saklar**, GitHub ise bu defterin ve proje kodunun sabit adresidir.

### Her bilgisayarda yapacağın işlem
1. Bu defteri [Google Colab'da](https://colab.research.google.com/github/konukailesi/iphone-video-fabrikasi/blob/main/colab/WanGP_Bulut_Ucretsiz_TR.ipynb) aç.
2. **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç (varsa).
3. **Çalışma zamanı → Tümünü çalıştır** seç. Drive erişim penceresinde kendi hesabına izin ver.
4. Son hücredeki **geçici gradio.live bağlantısını** açıp WanGP'yi kullan. Notebook oturumu açık kalmalıdır.

### Kalıcı saklama
- Varsayılan **USE_GOOGLE_DRIVE_DATA = True**; ilk çalıştırmada erişim izni istenir.
- Kalıcı klasör: `Google Drive / MyDrive / Wan2GP-data`; içeride `outputs`, `ckpts`, `loras`, `cache` klasörleri oluşturulur.
- Video dosyaları `MyDrive/Wan2GP-data/outputs` içinde saklanır. Google Drive kotası ve hız sınırlamaları uygulanır.
- GitHub *kodu* tutar, büyük model ve MP4 dosyalarını GitHub'a otomatik yüklemez.
- GPU ve uygulamanın kendisi **Colab oturumu kapanınca durur**. Başka PC'de açmak için yeni oturum başlatman gerekir; eski canlı Gradio bağlantısına güvenme.
- Her seferinde aynı anda tek Colab oturumunu kullan. Bilgisayar değiştirmek GPU'yu canlı taşımak değildir.

### Güvenlik ve maliyet
**Colab ücretsiz GPU garantisi yoktur; kota veya oturum sınırına takılabilir.** Gradio paylaşım bağlantısı herkese açık olabilir ve uygulama web kimlik doğrulaması kapalı başlayabilir. Bu bağlantıyı paylaşma ve özel görseller yüklemeden önce erişim güvenliğini değerlendir. Drive onayı yalnızca kendi Google hesabında verilir. Parola, token veya kimlik bilgilerini deftere yazma.

**İlk öneri:** 480p, kısa süre, düşük VRAM uyumlu FastWan. Kullanılabilen model Colab GPU ve güncel yazılıma göre değişir.

Kaynaklar: [WanGP](https://github.com/deepbeepmeep/WanGP) ve [Colab kaynak defteri](https://github.com/Square-Zero-Labs/Wan2GP-on-Colab).


# Wan2GP on Google Colab

Sets up [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) in a fresh GPU-backed Colab session.

Run the cells in order to prepare the runtime, install dependencies, and launch the Gradio interface. Click on the link in the output from the last cell to launch the app in your browser.

> **Colab VRAM note:** the free tier usually assigns a 15 GB T4 GPU. Most Wan2GP models exceed that budget; the Wan 2.2 TextImage2Video FastWan model works, producing roughly a 5 second 480p clip in about 8 minutes.

> **Tip:** lower the resolution in the Wan2GP interface before your first generation. The models default to 1280x720, which is more than a free Colab GPU handles comfortably.


## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

If this cell raises an error, go back to `Runtime → Change runtime type`, pick **GPU** and save.


In [9]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime → Change runtime type, select GPU (or TPU if GPUs are unavailable), save, then rerun this cell.'
    ) from exc


## 2. Configure the workspace path and optional persistent data storage


Set `USE_GOOGLE_DRIVE_DATA = True` if you want Google Drive to keep checkpoints, LoRAs, outputs and model caches across Colab restarts.


In [10]:
from pathlib import Path


# CHANGE THIS TO TRUE IF YOU WANT TO USE GOOGLE DRIVE FOR DATA STORAGE (PERSISTENT ACROSS SESSIONS)
USE_GOOGLE_DRIVE_DATA = True




DRIVE_MOUNT_POINT = Path('/content/drive')
WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
EPHEMERAL_DATA_ROOT = Path('/content/Wan2GP-data').resolve()
PERSISTENT_DATA_ROOT = (DRIVE_MOUNT_POINT / 'MyDrive' / 'Wan2GP-data').resolve()

if USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    WAN_DATA_ROOT = PERSISTENT_DATA_ROOT
    data_mode = 'Google Drive (persistent data)'
else:
    WAN_DATA_ROOT = EPHEMERAL_DATA_ROOT
    data_mode = 'Colab runtime disk (ephemeral data)'

WAN_CKPTS_DIR = (WAN_DATA_ROOT / 'ckpts').resolve()
WAN_LORAS_DIR = (WAN_DATA_ROOT / 'loras').resolve()
WAN_OUTPUTS_DIR = (WAN_DATA_ROOT / 'outputs').resolve()
# Local cache avoids Google Drive FUSE file lock errors
WAN_CACHE_DIR = (EPHEMERAL_DATA_ROOT / 'cache').resolve()
WAN_LTX2_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2').resolve()
WAN_LTX2_22B_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2_22B').resolve()

WAN2GP_ROOT.parent.mkdir(parents=True, exist_ok=True)
WAN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
WAN_CKPTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_22B_LORAS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Wan2GP repository path: {WAN2GP_ROOT}')
print(f'Data storage mode: {data_mode}')
print(f'Data root: {WAN_DATA_ROOT}')
print(f'Checkpoints: {WAN_CKPTS_DIR}')
print(f'LoRAs: {WAN_LORAS_DIR}')
print(f'LTX-2 LoRAs: {WAN_LTX2_LORAS_DIR}')
print(f'LTX-2 22B LoRAs: {WAN_LTX2_22B_LORAS_DIR}')
print(f'Outputs: {WAN_OUTPUTS_DIR}')
print(f'Cache: {WAN_CACHE_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wan2GP repository path: /content/Wan2GP
Data storage mode: Google Drive (persistent data)
Data root: /content/drive/MyDrive/Wan2GP-data
Checkpoints: /content/drive/MyDrive/Wan2GP-data/ckpts
LoRAs: /content/drive/MyDrive/Wan2GP-data/loras
LTX-2 LoRAs: /content/drive/MyDrive/Wan2GP-data/loras/ltx2
LTX-2 22B LoRAs: /content/drive/MyDrive/Wan2GP-data/loras/ltx2_22B
Outputs: /content/drive/MyDrive/Wan2GP-data/outputs
Cache: /content/Wan2GP-data/cache


## 3. Download or update Wan2GP

Clone the repository if it is not present yet; otherwise pull the latest changes.


In [11]:
import shutil, subprocess
from pathlib import Path

def merge_directory_contents(source_dir: Path, destination_dir: Path) -> None:
    for child in list(source_dir.iterdir()):
        destination = destination_dir / child.name
        if destination.exists():
            if child.is_dir() and destination.is_dir():
                merge_directory_contents(child, destination)
                child.rmdir()
                continue
            raise RuntimeError(f'Cannot move {child} into {destination_dir}: {destination} already exists.')
        shutil.move(str(child), str(destination))

def attach_data_directory(repo_path: Path, data_path: Path) -> None:
    data_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        if repo_path.resolve() != data_path.resolve():
            raise RuntimeError(f'{repo_path} already points to {repo_path.resolve()}, expected {data_path}.')
        print(f'Using existing link: {repo_path} -> {data_path}')
        return

    if repo_path.exists():
        if not repo_path.is_dir():
            raise RuntimeError(f'Expected a directory at {repo_path}.')
        merge_directory_contents(repo_path, data_path)
        repo_path.rmdir()
    else:
        repo_path.parent.mkdir(parents=True, exist_ok=True)

    repo_path.symlink_to(data_path, target_is_directory=True)
    print(f'Linked {repo_path.name} -> {data_path}')

MANAGED_PATHS = {'ckpts', 'loras', 'outputs', 'ffmpeg_bins'}

repo_url = 'https://github.com/deepbeepmeep/Wan2GP.git'
if WAN2GP_ROOT.exists():
    untracked = subprocess.run(
        ['git', '-C', str(WAN2GP_ROOT), 'ls-files', '--others', '--exclude-standard'],
        check=True,
        capture_output=True,
        text=True,
    )
    user_files = [
        line for line in untracked.stdout.splitlines()
        if line.split('/', 1)[0] not in MANAGED_PATHS
    ]
    if user_files:
        preview = ', '.join(user_files[:5]) + (' …' if len(user_files) > 5 else '')
        print(f'Repository has your own files in it ({preview}). Skipping the update to keep them.')
    else:
        print('Repository already exists. Updating to the latest version...')
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', '--', '.'], check=True)
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

attach_data_directory(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
attach_data_directory(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
attach_data_directory(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)


Repository already exists. Updating to the latest version...
Using existing link: /content/Wan2GP/ckpts -> /content/drive/MyDrive/Wan2GP-data/ckpts
Using existing link: /content/Wan2GP/loras -> /content/drive/MyDrive/Wan2GP-data/loras
Using existing link: /content/Wan2GP/outputs -> /content/drive/MyDrive/Wan2GP-data/outputs


## 4. Install system dependencies

Install shared libraries needed for video and audio processing, plus a compatible FFmpeg build when Colab's preinstalled version is too old. The FFmpeg download is integrity-checked and kept separate from system packages. If you see a warning about skipping an extra repository, it is safe to ignore.

In [12]:
import hashlib, json, os, platform, re, shutil, subprocess, tarfile, tempfile, urllib.request
from pathlib import Path

APT_PACKAGES = ['libglib2.0-0', 'libgl1', 'libportaudio2']
FFMPEG_RELEASE_API = 'https://api.github.com/repos/BtbN/FFmpeg-Builds/releases/latest'
FFMPEG_DOWNLOAD_PREFIX = 'https://github.com/BtbN/FFmpeg-Builds/releases/download/'
FFMPEG_BIN_DIR = WAN2GP_ROOT / 'ffmpeg_bins'
FFMPEG_CACHE_DIR = WAN_CACHE_DIR / 'downloads'
FFMPEG_ARCHITECTURES = {
    'x86_64': 'linux64',
    'amd64': 'linux64',
    'aarch64': 'linuxarm64',
    'arm64': 'linuxarm64',
}

def dpkg_installed(package: str) -> bool:
    return subprocess.run(['dpkg', '-s', package], capture_output=True).returncode == 0

missing = [package for package in APT_PACKAGES if not dpkg_installed(package)]
if missing:
    env = os.environ.copy()
    env['DEBIAN_FRONTEND'] = 'noninteractive'
    print('Installing:', ', '.join(missing))
    subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
    subprocess.run([
        'sudo', 'apt-get', 'install', '-y', '--no-install-recommends', *missing
    ], check=True, env=env)
else:
    print('System libraries already installed.')

def ffmpeg_supports_required_options(binary: Path) -> bool:
    try:
        result = subprocess.run(
            [str(binary), '-hide_banner', '-h', 'full'],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=30,
        )
    except (OSError, subprocess.TimeoutExpired):
        return False
    option_pattern = re.compile(r'^\s*-fps_mode(?:\[:[^]]+\])?(?:\s|$)', re.MULTILINE)
    return result.returncode == 0 and option_pattern.search(result.stdout) is not None

def executable_works(binary: Path) -> bool:
    try:
        return subprocess.run(
            [str(binary), '-version'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=30,
        ).returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        return False

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def github_json(url: str) -> dict:
    request = urllib.request.Request(
        url,
        headers={
            'Accept': 'application/vnd.github+json',
            'User-Agent': 'Wan2GP-Colab-notebook',
            'X-GitHub-Api-Version': '2022-11-28',
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)

def download_file(url: str, destination: Path, expected_size: int) -> None:
    request = urllib.request.Request(url, headers={'User-Agent': 'Wan2GP-Colab-notebook'})
    with urllib.request.urlopen(request, timeout=120) as response, destination.open('wb') as output:
        downloaded = 0
        next_report = 10
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
            downloaded += len(chunk)
            if expected_size:
                percent = downloaded * 100 // expected_size
                if percent >= next_report:
                    print(f'FFmpeg download: {min(percent, 100)}%')
                    next_report += 10
    if expected_size and downloaded != expected_size:
        raise RuntimeError(f'Incomplete FFmpeg download: expected {expected_size} bytes, got {downloaded}.')

def install_current_stable_ffmpeg() -> tuple[Path, Path]:
    machine = platform.machine().lower()
    asset_architecture = FFMPEG_ARCHITECTURES.get(machine)
    if asset_architecture is None:
        raise RuntimeError(f'No FFmpeg build is configured for the {machine} architecture.')
    asset_pattern = re.compile(
        rf'^ffmpeg-n(?P<version>\d+\.\d+)-latest-{asset_architecture}-gpl-(?P=version)\.tar\.xz$'
    )

    release = github_json(FFMPEG_RELEASE_API)
    candidates = []
    for asset in release.get('assets', []):
        match = asset_pattern.fullmatch(asset.get('name', ''))
        if match:
            version = tuple(int(part) for part in match.group('version').split('.'))
            candidates.append((version, asset))
    if not candidates:
        raise RuntimeError(f'No stable {asset_architecture} FFmpeg build was found in the upstream release.')

    _, asset = max(candidates, key=lambda item: item[0])
    asset_url = asset.get('browser_download_url', '')
    digest = asset.get('digest') or ''
    if not asset_url.startswith(FFMPEG_DOWNLOAD_PREFIX):
        raise RuntimeError(f'Unexpected FFmpeg download URL: {asset_url}')
    if not digest.startswith('sha256:'):
        raise RuntimeError('The FFmpeg release does not provide a SHA-256 digest; refusing an unverified download.')

    expected_sha256 = digest.split(':', 1)[1].lower()
    expected_size = int(asset.get('size') or 0)
    FFMPEG_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = FFMPEG_CACHE_DIR / asset['name']
    archive_valid = (
        archive_path.is_file()
        and (not expected_size or archive_path.stat().st_size == expected_size)
        and sha256_file(archive_path) == expected_sha256
    )
    if archive_valid:
        print(f'Using verified cached FFmpeg archive: {archive_path.name}')
    else:
        print(f'Downloading stable FFmpeg build: {asset["name"]}')
        with tempfile.NamedTemporaryFile(dir=FFMPEG_CACHE_DIR, suffix='.download', delete=False) as temp_file:
            temp_path = Path(temp_file.name)
        try:
            download_file(asset_url, temp_path, expected_size)
            actual_sha256 = sha256_file(temp_path)
            if actual_sha256 != expected_sha256:
                raise RuntimeError(
                    f'FFmpeg checksum mismatch: expected {expected_sha256}, got {actual_sha256}.'
                )
            os.replace(temp_path, archive_path)
        finally:
            temp_path.unlink(missing_ok=True)

    FFMPEG_BIN_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, mode='r:xz') as archive:
        members = {}
        for member in archive.getmembers():
            member_path = Path(member.name)
            if member.isfile() and member_path.parent.name == 'bin' and member_path.name in {'ffmpeg', 'ffprobe'}:
                if member_path.name in members:
                    raise RuntimeError(f'Duplicate {member_path.name} in FFmpeg archive.')
                members[member_path.name] = member
        if set(members) != {'ffmpeg', 'ffprobe'}:
            raise RuntimeError('The FFmpeg archive did not contain both ffmpeg and ffprobe.')
        for binary_name, member in members.items():
            source = archive.extractfile(member)
            if source is None:
                raise RuntimeError(f'Could not read {binary_name} from the FFmpeg archive.')
            destination = FFMPEG_BIN_DIR / binary_name
            temporary_destination = FFMPEG_BIN_DIR / f'.{binary_name}.tmp'
            with source, temporary_destination.open('wb') as output:
                shutil.copyfileobj(source, output)
            temporary_destination.chmod(0o755)
            os.replace(temporary_destination, destination)

    return FFMPEG_BIN_DIR / 'ffmpeg', FFMPEG_BIN_DIR / 'ffprobe'

local_ffmpeg = FFMPEG_BIN_DIR / 'ffmpeg'
local_ffprobe = FFMPEG_BIN_DIR / 'ffprobe'
system_ffmpeg_name = shutil.which('ffmpeg')
system_ffprobe_name = shutil.which('ffprobe')
system_ffmpeg = Path(system_ffmpeg_name) if system_ffmpeg_name else None
system_ffprobe = Path(system_ffprobe_name) if system_ffprobe_name else None

if local_ffmpeg.exists() or local_ffprobe.exists():
    if ffmpeg_supports_required_options(local_ffmpeg) and executable_works(local_ffprobe):
        ffmpeg_binary, ffprobe_binary = local_ffmpeg, local_ffprobe
        print('Using the compatible FFmpeg build already installed for Wan2GP.')
    else:
        print('The Wan2GP FFmpeg build is incomplete or outdated; updating it...')
        ffmpeg_binary, ffprobe_binary = install_current_stable_ffmpeg()
elif (
    system_ffmpeg is not None
    and system_ffprobe is not None
    and ffmpeg_supports_required_options(system_ffmpeg)
    and executable_works(system_ffprobe)
):
    ffmpeg_binary, ffprobe_binary = system_ffmpeg, system_ffprobe
    print("Using Colab's compatible system FFmpeg build.")
else:
    print("Colab's FFmpeg is missing or does not support -fps_mode; installing a current stable build...")
    ffmpeg_binary, ffprobe_binary = install_current_stable_ffmpeg()

if not ffmpeg_supports_required_options(ffmpeg_binary) or not executable_works(ffprobe_binary):
    raise RuntimeError('FFmpeg verification failed after installation.')

os.environ['FFMPEG_BINARY'] = str(ffmpeg_binary.resolve())
os.environ['FFPROBE_BINARY'] = str(ffprobe_binary.resolve())
binary_dir = str(ffmpeg_binary.resolve().parent)
path_parts = [part for part in os.environ.get('PATH', '').split(os.pathsep) if part != binary_dir]
os.environ['PATH'] = os.pathsep.join([binary_dir, *path_parts])
version_line = subprocess.run(
    [os.environ['FFMPEG_BINARY'], '-version'], check=True, capture_output=True, text=True
).stdout.splitlines()[0]
print(f'FFmpeg ready: {version_line}')
print(f'FFmpeg binary: {os.environ["FFMPEG_BINARY"]}')


Installing: libglib2.0-0
Using Colab's compatible system FFmpeg build.
FFmpeg ready: ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
FFmpeg binary: /usr/bin/ffmpeg


## 5. Python paket kurulumu — gerçek hata metni kırmızı alanda

**Bu sürümde UV hata nedeni kırmızı RuntimeError mesajının içine de ekleniyor.** Aynı hata olursa yalnızca kırmızı hatayı kopyalaman yeterli. Yeni sürümde `WANGP KURULUMUNUN GERCEK HATASI` ifadesini ara.


In [13]:
import json, os, subprocess, sys

PYTORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu128'
REPLACEMENT_TORCH = ('torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0')
UNSUPPORTED_TORCH = ('2.8.0', '2.9.')
ONNXRUNTIME_GPU = 'onnxruntime-gpu==1.22.0'

TORCH_PROBE = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import importlib.metadata as md, torch\n'
    '    for name in ("torch", "torchvision", "torchaudio"):\n'
    '        try:\n'
    '            info[name] = md.version(name)\n'
    '        except Exception:\n'
    '            pass\n'
    '    info["cuda"] = torch.cuda.is_available()\n'
    '    if info["cuda"]:\n'
    '        info["device"] = torch.cuda.get_device_name(0)\n'
    'except Exception:\n'
    '    pass\n'
    'print(json.dumps(info))\n'
)

ONNX_PROBE = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import onnxruntime\n'
    '    info["version"] = onnxruntime.__version__\n'
    '    info["providers"] = onnxruntime.get_available_providers()\n'
    'except Exception:\n'
    '    pass\n'
    'print(json.dumps(info))\n'
)

env = os.environ.copy()

# FIX 3: uv requires local filesystem locks, not Google Drive FUSE.
from pathlib import Path
uv_cache_dir = Path('/content/Wan2GP-runtime-cache/uv')
uv_cache_dir.mkdir(parents=True, exist_ok=True)
env['UV_CACHE_DIR'] = str(uv_cache_dir)
env['UV_LINK_MODE'] = 'copy'
os.environ['UV_CACHE_DIR'] = str(uv_cache_dir)
print('FIX 3 ACTIVE - local uv cache:', uv_cache_dir, flush=True)

def uv_pip(*args):
    # Print the actual uv resolver/build failure; CalledProcessError alone
    # hides the package name and cannot be diagnosed.
    cmd = [sys.executable, '-m', 'uv', 'pip', *args]
    print('Running:', ' '.join(cmd), flush=True)
    completed = subprocess.run(cmd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True,
                               env=env, check=False)
    if completed.stdout:
        print(completed.stdout[-16000:], flush=True)
    if completed.returncode:
        lines = (completed.stdout or '').splitlines()
        tail = '\n'.join(lines[-35:]) if lines else '(uv produced no output)'
        report_path = Path('/content/wanGP_step5_error.txt')
        report_path.write_text(
            'Command: ' + ' '.join(cmd) + '\n' +
            'Python: ' + sys.version.split()[0] + '\n' +
            'Torch: ' + str(probe(TORCH_PROBE).get('torch', 'unknown')) + '\n' +
            'uv output:\\n' + (completed.stdout or ''),
            encoding='utf-8'
        )
        raise RuntimeError(
            '\\n===== WANGP KURULUMUNUN GERCEK HATASI =====\\n' +
            tail +
            '\\n===== HATA SONU =====\\n' +
            'Log dosyasi: /content/wanGP_step5_error.txt'
        )


def uv_install(*args):
    uv_pip('install', '--system', *args)

def probe(code):
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, env=env)
    try:
        return json.loads(result.stdout.strip().splitlines()[-1])
    except Exception:
        return {}

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True, env=env)

torch_version = probe(TORCH_PROBE).get('torch', '')
try:
    torch_release = tuple(int(part) for part in torch_version.split('.')[:2])
except ValueError:
    torch_release = ()
torch_usable = (
    torch_release >= (2, 7)
    and not torch_version.startswith(UNSUPPORTED_TORCH)
)

if torch_usable:
    print(f'Using the PyTorch already installed in this runtime ({torch_version}).')
else:
    print('Installing PyTorch...')
    uv_install(*REPLACEMENT_TORCH, '--index-url', PYTORCH_INDEX_URL)

print('Installing Wan2GP packages...')
uv_install('-r', str(WAN2GP_ROOT / 'requirements.txt'), '--index-strategy', 'unsafe-best-match')

uv_install(ONNXRUNTIME_GPU)
onnx = probe(ONNX_PROBE)
if not onnx.get('version'):
    uv_pip('uninstall', '--system', 'onnxruntime-gpu')
    uv_install('onnxruntime>=1.22.0')
    onnx = probe(ONNX_PROBE)

report = probe(TORCH_PROBE)
if not report.get('torch'):
    raise RuntimeError(
        'PyTorch is not working after installation. Open Runtime -> Restart session, then run Steps 2 to 5 again.'
    )
if not report.get('cuda'):
    raise RuntimeError(
        'No GPU detected. Open Runtime -> Change runtime type, choose GPU, then run Steps 1 and 5 again.'
    )

print(f"Ready: PyTorch {report['torch']} on {report['device']}.")
if 'CUDAExecutionProvider' not in onnx.get('providers', []):
    print('Note: background removal will run on the CPU.')


FIX 3 ACTIVE - local uv cache: /content/Wan2GP-runtime-cache/uv
Using the PyTorch already installed in this runtime (2.14.1+cu130).
Installing Wan2GP packages...
Running: /usr/bin/python3 -m uv pip install --system -r /content/Wan2GP/requirements.txt --index-strategy unsafe-best-match
Using Python 3.13.16 environment at: /usr
Resolved 330 packages in 22.30s
Prepared 144 packages in 45.03s
Uninstalled 19 packages in 781ms
Installed 144 packages in 14.84s
 + addict==2.4.0
 - aiofiles==25.1.0
 + aiofiles==24.1.0
 + alembic==1.20.1
 + antlr4-python3-runtime==4.9.3
 + anyascii==0.3.3
 + apprise==1.12.0
 + asteroid-filterbanks==0.4.0
 + audio-separator==0.36.1
 + av==19.0.2
 - beartype==0.23.1
 + beartype==0.18.5
 + cfgv==3.5.0
 + chumpy==0.71 (from https://github.com/deepbeepmeep/chumpy/releases/download/v0.71/chumpy-0.71-py3-none-any.whl)
 + colorama==0.4.6
 + colorlog==6.12.0
 + comfy-kitchen==0.2.35
 + conformer==0.3.2
 + contractions==0.1.73
 + dashscope==2.0.4
 + ddgs==9.16.1
 + decord

## 5b. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend so Step 6 launches cleanly in Colab.


In [14]:
from pathlib import Path

# Replace the TkAgg backend with the headless Agg backend if present.
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


Replaced TkAgg with Agg in interact_tools.py.


## 6. WanGP'yi başlat — eksik optimum.quanto kontrolü

Önce 5. adımın tamamlanması gerekir. Bu hücre artık `optimum.quanto` eksikse `optimum-quanto==0.2.7` yüklemeyi dener ve import doğrulaması yapar. Diğer paketler 5. adımda yüklenmediyse farklı bağımlılık hataları oluşabilir; bu küçük onarım tüm gereksinimlerin kurulduğu garantisini vermez.


In [ ]:
import os, subprocess, sys, threading, time
from pathlib import Path

# Check the specific dependency that prevented the last launch.
# This does not claim the entire dependency installation succeeded.
def has_quanto():
    result = subprocess.run(
        [sys.executable, '-c', 'from optimum.quanto import QModuleMixin, register_qmodule'],
        capture_output=True, text=True)
    return result.returncode == 0

if not has_quanto():
    cache_dir = Path('/content/Wan2GP-runtime-cache/uv')
    cache_dir.mkdir(parents=True, exist_ok=True)
    install_env = os.environ.copy()
    install_env['UV_CACHE_DIR'] = str(cache_dir)
    install_env['UV_LINK_MODE'] = 'copy'
    print('Missing optimum.quanto; installing optimum-quanto==0.2.7...', flush=True)
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-input', 'optimum-quanto==0.2.7'],
        env=install_env, capture_output=True, text=True)
    print((result.stdout + '\n' + result.stderr)[-10000:])
    if result.returncode != 0 or not has_quanto():
        raise RuntimeError(
            'optimum-quanto could not be installed. Do not launch WanGP yet. '
            'Step 5 dependencies are incomplete; copy the pip error shown above.'
        )
    print('optimum.quanto dependency verified.', flush=True)
else:
    print('optimum.quanto dependency verified.', flush=True)

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')
cmd = [
    sys.executable,
    '-u',
    'wgp.py',
    '--listen',
    '--server-port', '7860',
    '--share',
    '--profile', '5',
]

if USE_GOOGLE_DRIVE_DATA:
    print('Using Google Drive for checkpoints, LoRAs, outputs and caches.')
else:
    print('Using Colab runtime storage for checkpoints, LoRAs, outputs and caches.')
print('Launching Wan2GP…')
process = subprocess.Popen(
    cmd,
    cwd=str(WAN2GP_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
stop_event = threading.Event()

def keepalive():
    while not stop_event.is_set():
        time.sleep(45)
        if stop_event.is_set():
            break
        print('[keepalive] Notebook cell still running…')

keepalive_thread = threading.Thread(target=keepalive, daemon=True)
keepalive_thread.start()

try:
    for line in iter(process.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping Wan2GP…')
    process.terminate()
finally:
    stop_event.set()
    process.wait()
    keepalive_thread.join(timeout=1)
    print(f'Wan2GP stopped (return code: {process.returncode}).')


optimum.quanto dependency verified.
Using Google Drive for checkpoints, LoRAs, outputs and caches.
Launching Wan2GP…
*********************** WanGP v19.25 - Copyright (c) 2024-2026 DeepBeepMeep ************************
****** Memory Management for the GPU Poor (mmgp 4.6.0) - Copyright (c) 2024-2026 DeepBeepMeep ******
[RAM] MMGP RAM Allocator: the RAM of freed CPU tensors goes back to the system
[VRAM] MMGP Optimized VRAM Allocator, spilling into RAM when VRAM is full
[keepalive] Notebook cell still running…
[keepalive] Notebook cell still running…
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[WanGP][Triton] Preparing _smoke_add_one_kernel (compiling, please wait)...
[WanGP][Triton] Compiled _smoke_add_one_kernel [fp32] in 3766 ms; configuration: num_warps=4 num_stages=3.
[WanGP][Triton] Preparing _smoke_block_pointer_kernel (compiling, please wait)